# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [17]:
import os
import json
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Initialization
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [19]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.

If a user ask to book the flight, call the tool set_ticket_price.

If the user ask to book/buy the flight after booking/buying ask them if they want anything else.
"""

In [20]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [21]:
tool_registry = {}

def tool(func):
    tool_registry[func.__name__] = func
    return func

In [22]:

@tool
def get_ticket_price(destination_city):
    print(f"DATABASE TOOL CALLED: Getting price for {destination_city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (destination_city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {destination_city} is ${result[0]}" if result else "No price data available for this city"

In [23]:
@tool
def reserve_ticket(destination_city, traveler_name):
    print(f"Tool reserve_ticket called for {destination_city} by {traveler_name}")
    price = get_ticket_price(destination_city)
    if price == "Unknown":
        return f"Sorry, we do not have ticket prices for {destination_city}."
    return f"Ticket to {destination_city.capitalize()} has been reserved for {traveler_name} at {price}."

In [24]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

reserve_function = {
    "name": "reserve_ticket",
    "description": "Reserve a ticket to the destination city for the traveler.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "traveler_name": {
                "type": "string",
                "description": "The name of the traveler",
            },
        },
        "required": ["destination_city", "traveler_name"],
        "additionalProperties": False
    }
}

In [25]:
tools = [{"type": "function", "function": price_function},
        {"type": "function", "function": reserve_function}]

In [26]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499", "paraguay": "$600"}


In [27]:
def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:
        func = tool_registry.get(tool_call.function.name)
        if not func:
            continue

        args = json.loads(tool_call.function.arguments)
        result = func(**args)

        responses.append({
            "role": "tool",
            "content": str(result),
            "tool_call_id": tool_call.id
        })
        
    return responses

In [28]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    reserve_ticket(city, price)

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()